# Standard KITTI 3D LiDAR training (Google Colab)

Notebook huấn luyện các biến thể BEV A0–A4. Chỉ sửa cell **Configuration**; mỗi run được khóa theo branch, commit và config hash trước khi resume.

In [ ]:
from google.colab import drive
from pathlib import Path
import json
import math
import os
import torch

# Main contains every registered BEV config.
BRANCH = "refractor_backbone"  # Or BRANCH = "main"
VARIANT = "BEVNEXT_LITEMLA"  # BEVNEXT_LITEMLA or A0–A4
CONFIG_OVERRIDE = None  # e.g. "configs/kitti/my_proposal.json"
RUN_NAME = None  # None creates a stable, resume-friendly name
SEED = 42
PRECISION = "bf16"  # fp32, fp16, bf16 (BF16 requires a supported GPU)
PHYSICAL_BATCH_SIZE = 2
ACCUMULATION_STEPS = 2
TARGET_BACKEND = "numba"  # numba is parity-tested; use python for eager reference runs
COMPILE_MODEL = False  # Opt in only after timing a full warm run
COMPILE_MODEL_ARGUMENT = "--compile-model" if COMPILE_MODEL else ""
RUNTIME_PROFILE = f"{TARGET_BACKEND}_{'compile' if COMPILE_MODEL else 'eager'}"

EPOCHS = 50  # Safe to increase when resuming the same run
NUM_WORKERS = max(0, min(6, (os.cpu_count() or 1) - 1))
RUN_SMOKE_TEST = True
SMOKE_TRAIN_BATCHES = 8
SMOKE_VAL_BATCHES = 4
RUN_EVALUATION = True
ALLOW_LEGACY_RESUME = False  # True only for runs created before this notebook

REPOSITORY_URL = "https://github.com/danhyoyo/Lidar.git"
REPO_DIR = Path("/content/Lidar")
KITTI_TAR_ROOT = Path("/content/drive/MyDrive/KITTI_DATASET_ZIP")
RAW_KITTI_ROOT = Path("/content/KITTI_DATASET")
PROCESSED_DATASET_DIR = REPO_DIR / "data/kitti/processed"
ARTIFACT_ROOT = Path("/content/drive/MyDrive/lidar_training_artifacts")

drive.mount("/content/drive")
%cd /content
!test -d Lidar || git clone "{REPOSITORY_URL}" Lidar
if _exit_code:
    raise RuntimeError("Không clone được repository.")
%cd /content/Lidar
!git fetch --prune origin "+refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}"
if _exit_code:
    raise RuntimeError(f"Không fetch được branch: {BRANCH}")
!git checkout --detach "origin/{BRANCH}"
if _exit_code:
    raise RuntimeError(f"Không checkout được branch: {BRANCH}")
COMMIT = !git rev-parse HEAD
COMMIT = COMMIT[0]
print(f"Branch: {BRANCH}\nCommit: {COMMIT}")

if not torch.cuda.is_available():
    raise RuntimeError("Bật GPU trong Runtime > Change runtime type trước khi train.")
if PRECISION == "bf16" and not torch.cuda.is_bf16_supported():
    raise RuntimeError("GPU này không hỗ trợ BF16; đổi PRECISION thành fp16 hoặc fp32.")
print(f"PyTorch: {torch.__version__}; GPU: {torch.cuda.get_device_name(0)}")

## Dependencies and variant

`CONFIG_OVERRIDE` is the escape hatch for a new proposal; it must be a repository-relative JSON path.

In [ ]:
%cd /content/Lidar
%pip install -q "numba>=0.59" shapely onnx tqdm

VARIANT_CONFIGS = {
    "BEVNEXT_LITEMLA": "configs/kitti/backbone_branch/kitti_bevnext_litemla.json",
    "A0": "configs/kitti/kitti_uwag_coordatt_aug.json",
    "A1": "configs/kitti/mobilebev/a1_legacy35_bev.json",
    "A2": "configs/kitti/mobilebev/a2_rich8_bev.json",
    "A3": "configs/kitti/mobilebev/a3_legacy35_sgfpn_bev.json",
    "A4": "configs/kitti/mobilebev/a4_rich8_sgfpn_bev.json",
}
if CONFIG_OVERRIDE is None and VARIANT not in VARIANT_CONFIGS:
    raise ValueError(f"Unknown VARIANT={VARIANT!r}; use CONFIG_OVERRIDE for a new proposal.")
CONFIG_RELATIVE = CONFIG_OVERRIDE or VARIANT_CONFIGS[VARIANT]
CONFIG = (REPO_DIR / CONFIG_RELATIVE).resolve()
if REPO_DIR not in CONFIG.parents or not CONFIG.is_file():
    raise FileNotFoundError(f"Config unavailable on {BRANCH}: {CONFIG_RELATIVE}")
EFFECTIVE_BATCH_SIZE = PHYSICAL_BATCH_SIZE * ACCUMULATION_STEPS
RUN_NAME = RUN_NAME or f"{VARIANT.lower()}_seed{SEED}_eb{EFFECTIVE_BATCH_SIZE}_{RUNTIME_PROFILE}"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Variant: {VARIANT}\nConfig: {CONFIG_RELATIVE}\nRun: {RUN_NAME}\nRuntime: {RUNTIME_PROFILE}")

## Prepare KITTI

The cell is idempotent: complete folders are reused; incomplete folders are re-extracted and validated.

In [ ]:
archives = {
    "velodyne": ("*.bin", 7481),
    "label_2": ("*.txt", 7481),
    "calib": ("*.txt", 7481),
}
RAW_KITTI_ROOT.mkdir(parents=True, exist_ok=True)

for folder, (pattern, expected_count) in archives.items():
    archive = KITTI_TAR_ROOT / f"{folder}.tar"
    extracted_dir = RAW_KITTI_ROOT / "training" / folder
    extracted_count = sum(1 for _ in extracted_dir.glob(pattern))

    if extracted_count != expected_count:
        if not archive.is_file():
            raise FileNotFoundError(f"Không tìm thấy archive: {archive}")
        archive_bytes = archive.stat().st_size
        !set -o pipefail; python3 -m tqdm --bytes --total {archive_bytes} --desc "Giải nén {folder}" < "{archive}" | tar --no-same-owner -xf - -C "{RAW_KITTI_ROOT}"
        if _exit_code:
            raise RuntimeError(f"Giải nén thất bại: {archive}")

    actual_count = sum(1 for _ in extracted_dir.glob(pattern))
    if actual_count != expected_count:
        raise RuntimeError(f"{folder}: {actual_count} file, cần {expected_count}")

pointcloud_dir = PROCESSED_DATASET_DIR / "pointcloud"
label_dir = PROCESSED_DATASET_DIR / "label"
dataset_ready = (
    sum(1 for _ in pointcloud_dir.glob("*.bin")) == 7481
    and sum(1 for _ in label_dir.glob("*.txt")) == 7481
    and (PROCESSED_DATASET_DIR / "train.txt").is_file()
    and (PROCESSED_DATASET_DIR / "val.txt").is_file()
)

if not dataset_ready:
    %cd /content/Lidar
    !python3 tools/kitti_training_pipeline/prepare_kitti.py \
      --kitti-root "{RAW_KITTI_ROOT}" \
      --output-root "{PROCESSED_DATASET_DIR}" \
      --config-output "{REPO_DIR / 'data/kitti/generated_kitti.json'}" \
      --train-ids "{REPO_DIR / 'splits/kitti/train.txt'}" \
      --val-ids "{REPO_DIR / 'splits/kitti/val.txt'}" \
      --pointcloud-mode symlink \
      --overwrite
    if _exit_code:
        raise RuntimeError("prepare_kitti.py thất bại")

pointcloud_count = sum(1 for _ in pointcloud_dir.glob("*.bin"))
label_count = sum(1 for _ in label_dir.glob("*.txt"))
assert pointcloud_count == label_count == 7481
assert (PROCESSED_DATASET_DIR / "train.txt").is_file()
assert (PROCESSED_DATASET_DIR / "val.txt").is_file()

print(f"Raw KITTI: {RAW_KITTI_ROOT}")
print(f"Processed: {PROCESSED_DATASET_DIR}")
print(f"Frames: {pointcloud_count}")


## Verify the checked-out proposal

Run this before the smoke/full training cells, so each branch proves its own test contract.

In [ ]:
%cd /content/Lidar
!MPLCONFIGDIR=/tmp/lidar-mpl python3 tests/test_mobile_bev.py
if _exit_code:
    raise RuntimeError("Proposal tests failed; do not train this checkout.")

## Acceleration-aware smoke test

This runs the selected Python/Numba and eager/compiled profile in a separate directory, then verifies its checkpoint, finite losses, and optimizer update before full training.

In [ ]:
SMOKE_RUN_NAME = f"{RUN_NAME}_smoke"
SMOKE_DIR = ARTIFACT_ROOT / SMOKE_RUN_NAME
SMOKE_METRICS = SMOKE_DIR / "metrics.jsonl"
SMOKE_CHECKPOINT = SMOKE_DIR / "checkpoints/last.pt"
if RUN_SMOKE_TEST:
    !python3 tools/kitti_training_pipeline/train.py --config "{CONFIG}" --detector-root detector --output-root "{ARTIFACT_ROOT}" --run-name "{SMOKE_RUN_NAME}" --device cuda --precision "{PRECISION}" --seed {SEED} --epochs 1 --physical-batch-size {PHYSICAL_BATCH_SIZE} --accumulation-steps {ACCUMULATION_STEPS} --max-train-batches {SMOKE_TRAIN_BATCHES} --max-val-batches {SMOKE_VAL_BATCHES} --num-workers {NUM_WORKERS} --target-backend "{TARGET_BACKEND}" {COMPILE_MODEL_ARGUMENT}
    if _exit_code:
        raise RuntimeError(f"Smoke test thất bại cho runtime {RUNTIME_PROFILE}.")
    if not SMOKE_CHECKPOINT.is_file() or not SMOKE_METRICS.is_file():
        raise RuntimeError("Smoke test không tạo đủ checkpoint/metrics.")
    smoke_rows = [line for line in SMOKE_METRICS.read_text(encoding="utf-8").splitlines() if line]
    smoke_row = json.loads(smoke_rows[-1]) if smoke_rows else None
    if not smoke_row or smoke_row["epoch"] != 1:
        raise RuntimeError("Smoke metrics thiếu epoch 1.")
    if smoke_row["optimizer_updates"] < 1:
        raise RuntimeError("Smoke test không thực hiện optimizer update.")
    if not math.isfinite(smoke_row["train_objective"]):
        raise RuntimeError("Smoke train objective không hữu hạn.")
    if not math.isfinite(smoke_row["validation"]["loss"]):
        raise RuntimeError("Smoke validation loss không hữu hạn.")
    print(f"Smoke PASS [{RUNTIME_PROFILE}]: {SMOKE_CHECKPOINT}")

## Full training / safe resume

The trainer writes `checkpoints/last.pt` after every completed epoch. Training stays attached to this cell; if interrupted, rerun it to resume from the latest completed epoch.

In [ ]:
import hashlib
import re

RUN_DIR = ARTIFACT_ROOT / RUN_NAME
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
TRAIN_LOG = RUN_DIR / "train.log"
RUN_METADATA_PATH = RUN_DIR / "run.json"
RUN_DIR.mkdir(parents=True, exist_ok=True)
RUN_METADATA = {"branch": BRANCH, "commit": COMMIT, "variant": VARIANT, "config_relative": CONFIG_RELATIVE, "config_sha256": hashlib.sha256(CONFIG.read_bytes()).hexdigest(), "seed": SEED, "precision": PRECISION, "physical_batch_size": PHYSICAL_BATCH_SIZE, "accumulation_steps": ACCUMULATION_STEPS, "target_backend": TARGET_BACKEND, "compile_model": COMPILE_MODEL, "num_workers": NUM_WORKERS}
if RUN_METADATA_PATH.is_file():
    if json.loads(RUN_METADATA_PATH.read_text(encoding="utf-8")) != RUN_METADATA:
        raise RuntimeError("Run metadata differs; choose a new RUN_NAME instead of resuming incompatible state.")
elif any(CHECKPOINT_DIR.glob("*.pt")) and not ALLOW_LEGACY_RESUME:
    raise RuntimeError("Existing checkpoints have no run.json. Set ALLOW_LEGACY_RESUME=True only after verifying compatibility.")
else:
    RUN_METADATA_PATH.write_text(json.dumps(RUN_METADATA, indent=2, sort_keys=True) + "\n", encoding="utf-8")

def checkpoint_epoch(path):
    if not path.is_file():
        return -1
    state = torch.load(path, map_location="cpu", weights_only=False)
    return int(state.get("epoch", -1)) if isinstance(state, dict) else -1

last_checkpoint = CHECKPOINT_DIR / "last.pt"
epoch_checkpoints = [path for path in CHECKPOINT_DIR.glob("*epoch.pt") if re.fullmatch(r"\d+epoch\.pt", path.name)]
resume_checkpoint = last_checkpoint if last_checkpoint.is_file() else max(epoch_checkpoints, key=checkpoint_epoch, default=None)
resume_epoch = checkpoint_epoch(resume_checkpoint) if resume_checkpoint else 0
if resume_epoch >= EPOCHS:
    print(f"Training already reached epoch {resume_epoch}/{EPOCHS}.")
else:
    RESUME_ARGUMENT = f'--resume "{resume_checkpoint}"' if resume_checkpoint else ""
    if resume_checkpoint:
        print(f"Resuming epoch {resume_epoch}: {resume_checkpoint}")
    !set -o pipefail; python3 -u tools/kitti_training_pipeline/train.py --config "{CONFIG}" --detector-root detector --output-root "{ARTIFACT_ROOT}" --run-name "{RUN_NAME}" --device cuda --precision "{PRECISION}" --seed {SEED} --epochs {EPOCHS} --physical-batch-size {PHYSICAL_BATCH_SIZE} --accumulation-steps {ACCUMULATION_STEPS} --num-workers {NUM_WORKERS} --target-backend "{TARGET_BACKEND}" {COMPILE_MODEL_ARGUMENT} {RESUME_ARGUMENT} 2>&1 | tee -a "{TRAIN_LOG}"
    if _exit_code:
        raise RuntimeError(f"Training failed with exit code {_exit_code}")

## Select checkpoint and evaluate

All variants use the trainer's minimum-validation-loss checkpoint and BEV AP evaluation.

In [ ]:
CHECKPOINT = RUN_DIR / "selected" / "best.pt"
if not CHECKPOINT.is_file():
    raise FileNotFoundError(CHECKPOINT)

if RUN_EVALUATION:
    split = REPO_DIR / "splits/kitti/val.txt"
    output = RUN_DIR / "evaluation_validation.json"
    !python3 tools/kitti_training_pipeline/evaluate_kitti_bev.py --name "{RUN_NAME}_validation" --backend pytorch --model "{CHECKPOINT}" --config "{CONFIG}" --detector-root detector --kitti-root "{RAW_KITTI_ROOT}" --split "{split}" --output "{output}" --device cuda --warmup-frames 10
    if _exit_code:
        raise RuntimeError("Đánh giá validation thất bại.")
print(f"Checkpoint: {CHECKPOINT}")